In [0]:
%run ./utils

In [0]:
from datetime import datetime, timedelta, timezone
import pyspark.sql.functions as F

def get_checksum_df(check_table, start_time, end_time, exclude_types=None):
    """
    查询指定时间窗口内的异常 consumer 明细。

    返回列:
        SRCC_ID, Market, Brand, SourceSystemCode, ConsumerId, Exclude_Type, TASK_ID
    """
    if exclude_types is None:
        exclude_types = DEFAULT_EXCLUDE_TYPES

    check_df = (
        spark.table(check_table)
        .filter(
            (F.col("SRCC_UPDATE_DT") >= F.lit(start_time)) &
            (F.col("SRCC_UPDATE_DT") <= F.lit(end_time))
        )
        .filter(F.col("EXCLUDE_TYPE").isin(exclude_types))
        .select(
            F.col("SRCC_ID"),
            F.col("SRCC_MRKT_CODE").alias("Market"),
            F.col("SRCC_BRND_CODE").alias("Brand"),
            F.col("SRCC_SRCS_CODE").alias("SourceSystemCode"),
            F.col("SRCC_CONSUMERID").alias("ConsumerId"),
            F.col("EXCLUDE_TYPE").alias("Exclude_Type"),
            F.col("task_id").alias("TASK_ID"),
        )
    )

    return check_df


In [0]:
def get_sample_by_exclude_type(check_df, sample_size):
    """
    按 Exclude_Type 分组，每种类型取 sample_size 条样本。

    使用 window row_number 在 Spark 端完成分组采样，避免把类型列表 collect 到 driver。

    返回:
        sample_df: 合并后的样本明细 DataFrame
    """
    from pyspark.sql import Window

    # 按 Exclude_Type 分区，取每组 SRCC_ID 最大的前 sample_size 条
    window = Window.partitionBy("Exclude_Type").orderBy(F.col("SRCC_ID").desc())
    sample_df = (
        check_df.withColumn("_row_num", F.row_number().over(window))
        .filter(F.col("_row_num") <= sample_size)
        .drop("_row_num")
    )

    return sample_df


In [0]:
def monitor_main(monitor_id, check_table, start_time, end_time, max_rows=MAX_ROWS, max_cols=MAX_COLS, to_addrs=None, exclude_types=None):
    # 1. query invalid data
    check_df = get_checksum_df(check_table, start_time, end_time, exclude_types)
    check_df.cache()

    invalid_data_count = check_df.count()

    if invalid_data_count > 0:
        print(f"This inspection found invalid data: {monitor_id}")
        print("Displaying invalid data summary:")
        display(check_df)

        # 2. build sample by Exclude_Type
        sample_df = get_sample_by_exclude_type(check_df, sample_size=max_rows)
        if sample_df is not None:
            print("Displaying sample data per Exclude_Type:")
            display(sample_df)

        # 3. build email body
        exclude_types_used = exclude_types or DEFAULT_EXCLUDE_TYPES
        sample_count = len(exclude_types_used) * max_rows
        html_body = (
            build_html_table_from_spark_df(sample_df, max_rows=sample_count, max_cols=max_cols)
            if sample_df is not None
            else "<p>No sample data available.</p>"
        )

        recipients = to_addrs or TO_ADDRS
        if not recipients:
            raise ValueError("to_addrs is empty; no recipients configured for the exception data report email.")

        send_email(
            subject=SUBJECT.format(yyyymmdd=end_time.strftime("%Y%m%d")),
            html_body=html_body,
            to_addrs=recipients,
            cc_addrs=CC_ADDRS,
            bcc_addrs=BCC_ADDRS,
            custom_text=f"This inspection found invalid data. <br>Check time period(UTC): {start_time} -&gt; {end_time}. <br>Check the table: {check_table}. <br>monitor_id: {monitor_id}"
        )

    else:
        print(f"There is no invalid data in this check: {monitor_id}")

    check_df.unpersist()


In [0]:
TO_ADDRS: List[str] = []
CC_ADDRS: List[str] = []
BCC_ADDRS: List[str] = []

SUBJECT = "[Warning] [MDM] Data Error {yyyymmdd}"
DEFAULT_EXCLUDE_TYPES = ["ExcludeByPkMiss", "FailByClearMiss"]

In [0]:
monitor_id = dbutils.widgets.get("monitor_id")
hour_time_period = int(dbutils.widgets.get("hour_time_period"))
check_table = dbutils.widgets.get("check_table")

try:
    trigger_timestamp_ms = int(dbutils.widgets.get("trigger_timestamp_ms")) / 1000
except:
    trigger_timestamp_ms = int(datetime.now().timestamp())

# Maximum rows/columns to show in the email HTML tables.
try:
    max_rows = int(dbutils.widgets.get("max_rows"))
except:
    max_rows = MAX_ROWS

try:
    max_cols = int(dbutils.widgets.get("max_cols"))
except:
    max_cols = MAX_COLS

# Comma-separated list of recipient email addresses.
to_addrs_str = dbutils.widgets.get("to_addrs")
to_addrs = [x.strip() for x in to_addrs_str.split(",") if x.strip()] if to_addrs_str else TO_ADDRS

end_time = datetime.fromtimestamp(trigger_timestamp_ms)
start_time = end_time - timedelta(hours=hour_time_period)

print(f"monitor_id: {monitor_id}")
print(f"check_table: {check_table}")
print(f"hour_time_period: {hour_time_period}")
print(f"max_rows: {max_rows}, max_cols: {max_cols}")
print(f"to_addrs: {to_addrs}")
print(f"start_time: {start_time}, end_time: {end_time}")

exclude_types_str = dbutils.widgets.get("exclude_types")
exclude_types = [x.strip() for x in exclude_types_str.split(",") if x.strip()] if exclude_types_str else None

print(f"exclude_types: {exclude_types}")

monitor_main(monitor_id, check_table, start_time, end_time, max_rows, max_cols, to_addrs, exclude_types)